# Chapter 4: Column Deep Dive

**Purpose:** Analyze each column in detail with distribution analysis, value validation, and transformation recommendations.

**What you'll learn:**
- How to validate value ranges for different column types
- How to interpret distribution shapes (skewness, kurtosis)
- When and why to apply transformations (log, sqrt, capping)
- How to detect zero-inflation and handle it

**Outputs:**
- Value range validation results
- Per-column distribution visualizations with statistics
- Skewness/kurtosis analysis with transformation recommendations
- Zero-inflation detection
- Type confirmation/override capability
- Updated exploration findings

## 4.1 Load Previous Findings

In [1]:
from customer_retention.analysis.notebook_progress import track_and_export_previous

track_and_export_previous("04_column_deep_dive.ipynb")

import numpy as np
import plotly.graph_objects as go
from scipy import stats

from customer_retention.analysis.auto_explorer import ExplorationFindings, RecommendationRegistry
from customer_retention.analysis.visualization import ChartBuilder, console, display_figure, display_table
from customer_retention.core.compat import native_pd, to_datetime
from customer_retention.core.config.column_config import ColumnType
from customer_retention.core.config.experiments import (
    FINDINGS_DIR,
)
from customer_retention.stages.profiling import (
    CategoricalDistributionAnalyzer,
    DistributionAnalyzer,
    TemporalAnalyzer,
    TemporalGranularity,
    TransformationType,
)
from customer_retention.stages.validation import DataValidator, RuleGenerator


In [2]:
from customer_retention.analysis.auto_explorer import load_notebook_findings

FINDINGS_PATH, _namespace, dataset_name = load_notebook_findings("04_column_deep_dive.ipynb")
print(f"Using: {FINDINGS_PATH}")

findings = ExplorationFindings.load(FINDINGS_PATH)
print(f"\nLoaded findings for {findings.column_count} columns from {findings.source_path}")

# Warn if this is event-level data (should run 01d first)
if findings.is_time_series and "_aggregated" not in FINDINGS_PATH:
    ts_meta = findings.time_series_metadata
    print("\n\u26a0\ufe0f  WARNING: This appears to be EVENT-LEVEL data")
    print(f"   Entity: {ts_meta.entity_column}, Time: {ts_meta.time_column}")
    print("   Recommendation: Run 01d_event_aggregation.ipynb first to create entity-level data")

Using: /Users/Vital/python/CustomerRetention/experiments/runs/3set-e48aeb77/datasets/customer_profiles/findings/customer_profiles_findings.yaml

Loaded findings for 7 columns from ../tests/fixtures/3set_customer_profiles.csv


## 4.2 Load Source Data

In [3]:
# Load data - handle aggregated data (parquet or Delta Lake)
from pathlib import Path

from customer_retention.analysis.auto_explorer.active_dataset_store import load_active_dataset
from customer_retention.stages.temporal import TEMPORAL_METADATA_COLS

# For aggregated data, load directly from the source path
if "_aggregated" in FINDINGS_PATH:
    source_path = Path(findings.source_path)
    if not source_path.is_absolute():
        source_path = Path("..") / source_path
    if source_path.is_dir():
        from customer_retention.integrations.adapters.factory import get_delta
        df = get_delta(force_local=True).read(str(source_path))
    elif source_path.is_file():
        df = native_pd.read_parquet(source_path)
    else:
        df = load_active_dataset(_namespace, dataset_name)
    data_source = f"aggregated:{source_path.name}"
else:
    # Standard loading for event-level or entity-level data
    df = load_active_dataset(_namespace, dataset_name)
    data_source = dataset_name

print(f"Loaded data from: {data_source}")
print(f"Shape: {df.shape}")

charts = ChartBuilder()

# Initialize recommendation registry for this exploration
registry = RecommendationRegistry()
registry.init_bronze(findings.source_path)

# Find target column for Gold layer initialization
target_col = next((name for name, col in findings.columns.items() if col.inferred_type == ColumnType.TARGET), None)
if target_col:
    registry.init_gold(target_col)

# Find entity column for Silver layer initialization
entity_col = next((name for name, col in findings.columns.items() if col.inferred_type == ColumnType.IDENTIFIER), None)
if entity_col:
    registry.init_silver(entity_col)

print(f"Initialized recommendation registry (Bronze: {findings.source_path})")


Loaded data from: customer_profiles
Shape: (1000, 17)
Initialized recommendation registry (Bronze: ../tests/fixtures/3set_customer_profiles.csv)


## 4.3 Value Range Validation

**📖 Interpretation Guide:**
- **Percentage fields** (rates): Should be 0-100 or 0-1 depending on format
- **Binary fields**: Should only contain 0 and 1
- **Count fields**: Should be non-negative integers
- **Amount fields**: Should be non-negative (unless refunds are possible)

**What to Watch For:**
- Rates > 100% suggest measurement or data entry errors
- Negative values in fields that should be positive
- Binary fields with values other than 0/1

**Actions:**
- Cap rates at 100 if they exceed (or investigate cause)
- Flag records with impossible negative values
- Convert binary fields to proper 0/1 encoding

In [4]:
validator = DataValidator()
range_rules = RuleGenerator.from_findings(findings)

console.start_section()
console.header("Value Range Validation")

if range_rules:
    range_results = validator.validate_value_ranges(df, range_rules)

    issues_found = []
    for r in range_results:
        detail = f"{r.invalid_values} invalid" if r.invalid_values > 0 else None
        console.check(f"{r.column_name} ({r.rule_type})", r.invalid_values == 0, detail)
        if r.invalid_values > 0:
            issues_found.append(r)

    all_invalid = sum(r.invalid_values for r in range_results)
    if all_invalid == 0:
        console.success("All value ranges valid")
    else:
        console.error(f"Found {all_invalid:,} values outside expected ranges")

        console.info("Examples of invalid values:")
        for r in issues_found[:3]:
            col = r.column_name
            if col in df.columns:
                if r.rule_type == 'binary':
                    invalid_mask = ~df[col].isin([0, 1, np.nan])
                    condition = "value not in [0, 1]"
                elif r.rule_type == 'non_negative':
                    invalid_mask = df[col] < 0
                    condition = "value < 0"
                elif r.rule_type == 'percentage':
                    invalid_mask = (df[col] < 0) | (df[col] > 100)
                    condition = "value < 0 or value > 100"
                elif r.rule_type == 'rate':
                    invalid_mask = (df[col] < 0) | (df[col] > 1)
                    condition = "value < 0 or value > 1"
                else:
                    continue

                invalid_values = df.loc[invalid_mask, col].dropna()
                if len(invalid_values) > 0:
                    examples = invalid_values.head(5).tolist()
                    console.metric(f"  {col}", f"{examples}")

                    # Add filtering recommendation
                    registry.add_bronze_filtering(
                        column=col, condition=condition, action="cap",
                        rationale=f"{r.invalid_values} values violate {r.rule_type} constraint",
                        source_notebook="04_column_deep_dive"
                    )

    console.info("Rules auto-generated from detected column types")
else:
    range_results = []
    console.info("No validation rules generated - no binary/numeric columns detected")

console.end_section()

#### VALUE RANGE VALIDATION  
[OK] is_missing_contract_start (binary)  
[OK] is_future_contract_start (binary)  
[OK] All value ranges valid  
*(i) Rules auto-generated from detected column types*

## 4.4 Numeric Columns Analysis

**📖 How to Interpret These Charts:**
- **Red dashed line** = Mean (sensitive to outliers)
- **Green solid line** = Median (robust to outliers)
- **Large gap between mean and median** = Skewed distribution
- **Long right tail** = Positive skew (common in count/amount data)

**📖 Understanding Distribution Metrics**

| Metric | Interpretation | Action |
|--------|---------------|--------|
| **Skewness** | Measures asymmetry | \|skew\| > 1: Consider log transform |
| **Kurtosis** | Measures tail heaviness | kurt > 10: Cap outliers before transform |
| **Zero %** | Percentage of zeros | > 40%: Use zero-inflation handling |

**📖 Transformation Decision Tree:**
1. If zeros > 40% → Create binary indicator + log(non-zeros)
2. If \|skewness\| > 1 AND kurtosis > 10 → Cap then log
3. If \|skewness\| > 1 → Log transform
4. If kurtosis > 10 → Cap outliers only
5. Otherwise → Standard scaling is sufficient

In [5]:
analyzer = DistributionAnalyzer()

numeric_cols = [
    name for name, col in findings.columns.items()
    if col.inferred_type in [ColumnType.NUMERIC_CONTINUOUS, ColumnType.NUMERIC_DISCRETE]
    and name not in TEMPORAL_METADATA_COLS
]

analyses = analyzer.analyze_dataframe(df, numeric_cols)
recommendations = {col: analyzer.recommend_transformation(analysis)
                   for col, analysis in analyses.items()}

for col_name in numeric_cols:
    col_info = findings.columns[col_name]
    analysis = analyses.get(col_name)
    rec = recommendations.get(col_name)

    print(f"\n{'='*70}")
    print(f"Column: {col_name}")
    print(f"Type: {col_info.inferred_type.value} (Confidence: {col_info.confidence:.0%})")
    print("-" * 70)

    if analysis:
        print("\U0001f4ca Distribution Statistics:")
        print(f"   Mean: {analysis.mean:.3f}  |  Median: {analysis.median:.3f}  |  Std: {analysis.std:.3f}")
        print(f"   Range: [{analysis.min_value:.3f}, {analysis.max_value:.3f}]")
        if analysis.percentiles:
            print(f"   Percentiles: 1%={analysis.percentiles.get('p1', 0):.3f}, 25%={analysis.q1:.3f}, 75%={analysis.q3:.3f}, 99%={analysis.percentiles.get('p99', 0):.3f}")
        print("\n\U0001f4c8 Shape Analysis:")
        skew_label = '(Right-skewed)' if analysis.skewness > 0.5 else '(Left-skewed)' if analysis.skewness < -0.5 else '(Symmetric)'
        print(f"   Skewness: {analysis.skewness:.2f} {skew_label}")
        kurt_label = '(Heavy tails/outliers)' if analysis.kurtosis > 3 else '(Light tails)'
        print(f"   Kurtosis: {analysis.kurtosis:.2f} {kurt_label}")
        print(f"   Zeros: {analysis.zero_count:,} ({analysis.zero_percentage:.1f}%)")
        print(f"   Outliers (IQR): {analysis.outlier_count_iqr:,} ({analysis.outlier_percentage:.1f}%)")

        if rec:
            print(f"\n\U0001f527 Recommended Transformation: {rec.recommended_transform.value}")
            print(f"   Reason: {rec.reason}")
            print(f"   Priority: {rec.priority}")
            if rec.warnings:
                for warn in rec.warnings:
                    print(f"   \u26a0\ufe0f {warn}")

    data_np = df[col_name].dropna().to_numpy()
    fig = go.Figure()

    fig.add_trace(go.Histogram(x=data_np, nbinsx=50, name='Distribution',
                                marker_color='steelblue', opacity=0.7))

    mean_val = analysis.mean
    median_val = analysis.median

    mean_position = "top right" if mean_val >= median_val else "top left"
    median_position = "top left" if mean_val >= median_val else "top right"

    fig.add_vline(
        x=mean_val, line_dash="dash", line_color="red",
        annotation_text=f"Mean: {mean_val:.2f}", annotation_position=mean_position,
        annotation_font_color="red", annotation_bgcolor="rgba(255,255,255,0.8)"
    )
    fig.add_vline(
        x=median_val, line_dash="solid", line_color="green",
        annotation_text=f"Median: {median_val:.2f}", annotation_position=median_position,
        annotation_font_color="green", annotation_bgcolor="rgba(255,255,255,0.8)"
    )

    if analysis and analysis.outlier_percentage > 5 and analysis.percentiles.get('p99') is not None:
        fig.add_vline(x=analysis.percentiles['p99'], line_dash="dot", line_color="orange",
                      annotation_text=f"99th: {analysis.percentiles['p99']:.2f}",
                      annotation_position="top right",
                      annotation_font_color="orange",
                      annotation_bgcolor="rgba(255,255,255,0.8)")

    transform_label = rec.recommended_transform.value if rec else "none"
    fig.update_layout(
        title=f"Distribution: {col_name}<br><sub>Skew: {analysis.skewness:.2f} | Kurt: {analysis.kurtosis:.2f} | Strategy: {transform_label}</sub>",
        xaxis_title=col_name, yaxis_title="Count",
        template='plotly_white', height=400
    )
    display_figure(fig)


Column: contract_start_delta_hours
Type: numeric_continuous (Confidence: 100%)
----------------------------------------------------------------------
📊 Distribution Statistics:
   Mean: -8760.000  |  Median: -8760.000  |  Std: 0.000
   Range: [-8760.000, -8760.000]
   Percentiles: 1%=-8760.000, 25%=-8760.000, 75%=-8760.000, 99%=-8760.000

📈 Shape Analysis:
   Skewness: 0.00 (Symmetric)
   Kurtosis: 0.00 (Light tails)
   Zeros: 0 (0.0%)
   Outliers (IQR): 0 (0.0%)

🔧 Recommended Transformation: none
   Reason: Distribution is approximately normal (skewness: 0.00)
   Priority: low



Column: contract_start_hour
Type: numeric_continuous (Confidence: 100%)
----------------------------------------------------------------------
📊 Distribution Statistics:
   Mean: 11.387  |  Median: 11.500  |  Std: 6.834
   Range: [0.000, 23.000]
   Percentiles: 1%=0.000, 25%=5.000, 75%=17.000, 99%=23.000

📈 Shape Analysis:
   Skewness: 0.02 (Symmetric)
   Kurtosis: -1.20 (Light tails)
   Zeros: 32 (3.2%)
   Outliers (IQR): 0 (0.0%)

🔧 Recommended Transformation: none
   Reason: Distribution is approximately normal (skewness: 0.02)
   Priority: low



Column: contract_start_dow
Type: numeric_continuous (Confidence: 100%)
----------------------------------------------------------------------
📊 Distribution Statistics:
   Mean: 2.988  |  Median: 3.000  |  Std: 1.966
   Range: [0.000, 6.000]
   Percentiles: 1%=0.000, 25%=1.000, 75%=5.000, 99%=6.000

📈 Shape Analysis:
   Skewness: 0.02 (Symmetric)
   Kurtosis: -1.20 (Light tails)
   Zeros: 134 (13.4%)
   Outliers (IQR): 0 (0.0%)

🔧 Recommended Transformation: none
   Reason: Distribution is approximately normal (skewness: 0.02)
   Priority: low



Column: contract_start_is_weekend
Type: numeric_continuous (Confidence: 100%)
----------------------------------------------------------------------
📊 Distribution Statistics:
   Mean: 0.273  |  Median: 0.000  |  Std: 0.446
   Range: [0.000, 1.000]
   Percentiles: 1%=0.000, 25%=0.000, 75%=1.000, 99%=1.000

📈 Shape Analysis:
   Skewness: 1.02 (Right-skewed)
   Kurtosis: -0.96 (Light tails)
   Zeros: 727 (72.7%)
   Outliers (IQR): 0 (0.0%)

🔧 Recommended Transformation: zero_inflation_handling
   Reason: Significant zero-inflation (72.7%)
   Priority: medium
   ⚠️ Many zero values may indicate a mixture distribution



Column: days_since_contract_start
Type: numeric_continuous (Confidence: 100%)
----------------------------------------------------------------------
📊 Distribution Statistics:
   Mean: 365.000  |  Median: 365.000  |  Std: 0.000
   Range: [365.000, 365.000]
   Percentiles: 1%=365.000, 25%=365.000, 75%=365.000, 99%=365.000

📈 Shape Analysis:
   Skewness: 0.00 (Symmetric)
   Kurtosis: 0.00 (Light tails)
   Zeros: 0 (0.0%)
   Outliers (IQR): 0 (0.0%)

🔧 Recommended Transformation: none
   Reason: Distribution is approximately normal (skewness: 0.00)
   Priority: low



Column: days_until_contract_start
Type: numeric_continuous (Confidence: 100%)
----------------------------------------------------------------------
📊 Distribution Statistics:
   Mean: -365.000  |  Median: -365.000  |  Std: 0.000
   Range: [-365.000, -365.000]
   Percentiles: 1%=-365.000, 25%=-365.000, 75%=-365.000, 99%=-365.000

📈 Shape Analysis:
   Skewness: 0.00 (Symmetric)
   Kurtosis: 0.00 (Light tails)
   Zeros: 0 (0.0%)
   Outliers (IQR): 0 (0.0%)

🔧 Recommended Transformation: none
   Reason: Distribution is approximately normal (skewness: 0.00)
   Priority: low



Column: log1p_days_since_contract_start
Type: numeric_continuous (Confidence: 100%)
----------------------------------------------------------------------
📊 Distribution Statistics:
   Mean: 5.903  |  Median: 5.903  |  Std: 0.000
   Range: [5.903, 5.903]
   Percentiles: 1%=5.903, 25%=5.903, 75%=5.903, 99%=5.903

📈 Shape Analysis:
   Skewness: 0.00 (Symmetric)
   Kurtosis: 0.00 (Light tails)
   Zeros: 0 (0.0%)
   Outliers (IQR): 0 (0.0%)

🔧 Recommended Transformation: none
   Reason: Distribution is approximately normal (skewness: 0.00)
   Priority: low


In [6]:
if numeric_cols and analyses:
    stats_data = []
    for col_name in numeric_cols:
        analysis = analyses.get(col_name)
        if analysis and analysis.count > 0:
            stats_data.append({
                "feature": col_name,
                "count": analysis.count,
                "mean": analysis.mean,
                "std": analysis.std,
                "min": analysis.min_value,
                "25%": analysis.q1,
                "50%": analysis.median,
                "75%": analysis.q3,
                "95%": analysis.percentiles.get("p95", 0),
                "99%": analysis.percentiles.get("p99", 0),
                "max": analysis.max_value,
                "skewness": analysis.skewness,
                "kurtosis": analysis.kurtosis,
            })

    if stats_data:
        stats_df = native_pd.DataFrame(stats_data)

        display_stats = stats_df.copy()
        for col in ["mean", "std", "min", "25%", "50%", "75%", "95%", "99%", "max"]:
            display_stats[col] = display_stats[col].apply(lambda x: f"{x:.3f}")
        display_stats["skewness"] = display_stats["skewness"].apply(lambda x: f"{x:.3f}")
        display_stats["kurtosis"] = display_stats["kurtosis"].apply(lambda x: f"{x:.3f}")

        print("=" * 80)
        print("NUMERICAL FEATURE STATISTICS")
        print("=" * 80)
        display_table(display_stats)
    else:
        print("No numeric columns with data to display (all counts are 0)")

NUMERICAL FEATURE STATISTICS


feature,count,mean,std,min,25%,50%,75%,95%,99%,max,skewness,kurtosis
contract_start_delta_hours,1000,-8760.000,0.000,-8760.000,-8760.000,-8760.000,-8760.000,-8760.000,-8760.000,-8760.000,0.000,0.000
contract_start_hour,1000,11.387,6.834,0.000,5.000,11.500,17.000,22.000,23.000,23.000,0.018,-1.202
contract_start_dow,1000,2.988,1.966,0.000,1.000,3.000,5.000,6.000,6.000,6.000,0.023,-1.204
contract_start_is_weekend,1000,0.273,0.446,0.000,0.000,0.000,1.000,1.000,1.000,1.000,1.021,-0.960
days_since_contract_start,1000,365.000,0.000,365.000,365.000,365.000,365.000,365.000,365.000,365.000,0.000,0.000
days_until_contract_start,1000,-365.000,0.000,-365.000,-365.000,-365.000,-365.000,-365.000,-365.000,-365.000,0.000,0.000
log1p_days_since_contract_start,1000,5.903,0.000,5.903,5.903,5.903,5.903,5.903,5.903,5.903,0.000,0.000


## 4.5 Distribution Summary & Transformation Plan

This table summarizes all numeric columns with their recommended transformations.

In [7]:
# Build transformation summary table
summary_data = []
for col_name in numeric_cols:
    analysis = analyses.get(col_name)
    rec = recommendations.get(col_name)

    if analysis and rec:
        summary_data.append({
            "Column": col_name,
            "Skewness": f"{analysis.skewness:.2f}",
            "Kurtosis": f"{analysis.kurtosis:.2f}",
            "Zeros %": f"{analysis.zero_percentage:.1f}%",
            "Outliers %": f"{analysis.outlier_percentage:.1f}%",
            "Transform": rec.recommended_transform.value,
            "Priority": rec.priority
        })

        # Add Gold transformation recommendation if not "none"
        if rec.recommended_transform != TransformationType.NONE and registry.gold:
            registry.add_gold_transformation(
                column=col_name,
                transform=rec.recommended_transform.value,
                parameters=rec.parameters,
                rationale=rec.reason,
                source_notebook="04_column_deep_dive"
            )

if summary_data:
    summary_df = native_pd.DataFrame(summary_data)
    display_table(summary_df)

    # Show how many transformation recommendations were added
    transform_count = sum(1 for r in recommendations.values() if r and r.recommended_transform != TransformationType.NONE)
    if transform_count > 0 and registry.gold:
        print(f"\n✅ Added {transform_count} transformation recommendations to Gold layer")
else:
    console.info("No numeric columns to summarize")

Column,Skewness,Kurtosis,Zeros %,Outliers %,Transform,Priority
contract_start_delta_hours,0.00,0.00,0.0%,0.0%,none,low
contract_start_hour,0.02,-1.20,3.2%,0.0%,none,low
contract_start_dow,0.02,-1.20,13.4%,0.0%,none,low
contract_start_is_weekend,1.02,-0.96,72.7%,0.0%,zero_inflation_handling,medium
days_since_contract_start,0.00,0.00,0.0%,0.0%,none,low
days_until_contract_start,0.00,0.00,0.0%,0.0%,none,low
log1p_days_since_contract_start,0.00,0.00,0.0%,0.0%,none,low



✅ Added 1 transformation recommendations to Gold layer


## 4.6 Categorical Columns Analysis

**📖 Distribution Metrics (Analogues to Numeric Skewness/Kurtosis):**

| Metric | Interpretation | Action |
|--------|---------------|--------|
| **Imbalance Ratio** | Largest / Smallest category count | > 10: Consider grouping rare categories |
| **Entropy** | Diversity measure (0 = one category, higher = more uniform) | Low entropy: May need stratified sampling |
| **Top-3 Concentration** | % of data in top 3 categories | > 90%: Rare categories may cause issues |
| **Rare Category %** | Categories with < 1% of data | High %: Group into "Other" category |

**📖 Encoding Recommendations:**
- **Low cardinality (≤5)** → One-hot encoding
- **Medium cardinality (6-20)** → One-hot or Target encoding
- **High cardinality (>20)** → Target encoding or Frequency encoding
- **Cyclical (days, months)** → Sin/Cos encoding

**⚠️ Common Issues:**
- Rare categories can cause overfitting with one-hot encoding
- High cardinality + one-hot = feature explosion
- Imbalanced categories may need special handling in train/test splits

In [8]:
# Use framework's CategoricalDistributionAnalyzer
cat_analyzer = CategoricalDistributionAnalyzer()

categorical_cols = [
    name for name, col in findings.columns.items()
    if col.inferred_type in [ColumnType.CATEGORICAL_NOMINAL, ColumnType.CATEGORICAL_ORDINAL, ColumnType.CATEGORICAL_CYCLICAL]
    and col.inferred_type != ColumnType.TEXT  # TEXT columns processed separately in 02a
    and name not in TEMPORAL_METADATA_COLS
]

# Analyze all categorical columns
cat_analyses = cat_analyzer.analyze_dataframe(df, categorical_cols)

# Get encoding recommendations
cyclical_cols = [name for name, col in findings.columns.items()
                 if col.inferred_type == ColumnType.CATEGORICAL_CYCLICAL]
cat_recommendations = cat_analyzer.get_all_recommendations(df, categorical_cols, cyclical_columns=cyclical_cols)

for col_name in categorical_cols:
    col_info = findings.columns[col_name]
    analysis = cat_analyses.get(col_name)
    rec = next((r for r in cat_recommendations if r.column_name == col_name), None)

    print(f"\n{'='*70}")
    print(f"Column: {col_name}")
    print(f"Type: {col_info.inferred_type.value} (Confidence: {col_info.confidence:.0%})")
    print("-" * 70)

    if analysis:
        print("\n📊 Distribution Metrics:")
        print(f"   Categories: {analysis.category_count}")
        print(f"   Imbalance Ratio: {analysis.imbalance_ratio:.1f}x (largest/smallest)")
        print(f"   Entropy: {analysis.entropy:.2f} ({analysis.normalized_entropy*100:.0f}% of max)")
        print(f"   Top-1 Concentration: {analysis.top1_concentration:.1f}%")
        print(f"   Top-3 Concentration: {analysis.top3_concentration:.1f}%")
        print(f"   Rare Categories (<1%): {analysis.rare_category_count}")

        # Interpretation
        print("\n📈 Interpretation:")
        if analysis.has_low_diversity:
            print("   ⚠️ LOW DIVERSITY: Distribution dominated by few categories")
        elif analysis.normalized_entropy > 0.9:
            print("   ✓ HIGH DIVERSITY: Categories are relatively balanced")
        else:
            print("   ✓ MODERATE DIVERSITY: Some category dominance but acceptable")

        if analysis.imbalance_ratio > 100:
            print("   🔴 SEVERE IMBALANCE: Rarest category has very few samples")
        elif analysis.is_imbalanced:
            print("   🟡 MODERATE IMBALANCE: Consider grouping rare categories")

        # Recommendations
        if rec:
            print("\n🔧 Recommendations:")
            print(f"   Encoding: {rec.encoding_type.value}")
            print(f"   Reason: {rec.reason}")
            print(f"   Priority: {rec.priority}")

            if rec.preprocessing_steps:
                print("   Preprocessing:")
                for step in rec.preprocessing_steps:
                    print(f"      • {step}")

            if rec.warnings:
                for warn in rec.warnings:
                    print(f"   ⚠️ {warn}")

    # Visualization
    value_counts = df[col_name].value_counts()
    subtitle = f"Entropy: {analysis.normalized_entropy*100:.0f}% | Imbalance: {analysis.imbalance_ratio:.1f}x | Rare: {analysis.rare_category_count}" if analysis else ""
    fig = charts.bar_chart(
        value_counts.head(10).index.tolist(),
        value_counts.head(10).values.tolist(),
        title=f"Top Categories: {col_name}<br><sub>{subtitle}</sub>"
    )
    display_figure(fig)

# Summary table and add recommendations to registry
if cat_analyses:
    print("\n" + "=" * 70)
    print("CATEGORICAL COLUMNS SUMMARY")
    print("=" * 70)
    summary_data = []
    for col_name, analysis in cat_analyses.items():
        rec = next((r for r in cat_recommendations if r.column_name == col_name), None)
        summary_data.append({
            "Column": col_name,
            "Categories": analysis.category_count,
            "Imbalance": f"{analysis.imbalance_ratio:.1f}x",
            "Entropy": f"{analysis.normalized_entropy*100:.0f}%",
            "Top-3 Conc.": f"{analysis.top3_concentration:.1f}%",
            "Rare (<1%)": analysis.rare_category_count,
            "Encoding": rec.encoding_type.value if rec else "N/A"
        })

        # Add encoding recommendation to Gold layer
        if rec and registry.gold:
            registry.add_gold_encoding(
                column=col_name,
                method=rec.encoding_type.value,
                rationale=rec.reason,
                source_notebook="04_column_deep_dive"
            )

    display_table(native_pd.DataFrame(summary_data))

    if registry.gold:
        print(f"\n✅ Added {len(cat_recommendations)} encoding recommendations to Gold layer")


Column: activity_tier
Type: categorical_nominal (Confidence: 90%)
----------------------------------------------------------------------

📊 Distribution Metrics:
   Categories: 3
   Imbalance Ratio: 3.0x (largest/smallest)
   Entropy: 1.46 (92% of max)
   Top-1 Concentration: 45.0%
   Top-3 Concentration: 100.0%
   Rare Categories (<1%): 0

📈 Interpretation:
   ✓ HIGH DIVERSITY: Categories are relatively balanced

🔧 Recommendations:
   Encoding: one_hot
   Reason: Low cardinality (3 categories) - safe feature expansion
   Priority: low



Column: risk_tier
Type: categorical_nominal (Confidence: 90%)
----------------------------------------------------------------------

📊 Distribution Metrics:
   Categories: 3
   Imbalance Ratio: 4.9x (largest/smallest)
   Entropy: 1.36 (86% of max)
   Top-1 Concentration: 54.9%
   Top-3 Concentration: 100.0%
   Rare Categories (<1%): 0

📈 Interpretation:
   ✓ MODERATE DIVERSITY: Some category dominance but acceptable

🔧 Recommendations:
   Encoding: one_hot
   Reason: Low cardinality (3 categories) - safe feature expansion
   Priority: low



CATEGORICAL COLUMNS SUMMARY


Column,Categories,Imbalance,Entropy,Top-3 Conc.,Rare (<1%),Encoding
activity_tier,3,3.0x,92%,100.0%,0,one_hot
risk_tier,3,4.9x,86%,100.0%,0,one_hot



✅ Added 2 encoding recommendations to Gold layer


## 4.7 Datetime Columns Analysis

**📖 Unlike numeric transformations, datetime analysis recommends NEW FEATURES to create:**

| Recommendation Type | Purpose | Examples |
|---------------------|---------|----------|
| **Feature Engineering** | Create predictive features from dates | `days_since_signup`, `tenure_years`, `month_sin_cos` |
| **Modeling Strategy** | How to structure train/test | Time-based splits when trends detected |
| **Data Quality** | Issues to address before modeling | Placeholder dates (1/1/1900) to filter |

**📖 Feature Engineering Strategies:**
- **Recency**: `days_since_X` - How recent was the event? (useful for predicting behavior)
- **Tenure**: `tenure_years` - How long has customer been active? (maturity/loyalty)
- **Duration**: `days_between_A_and_B` - Time between events (e.g., signup to first purchase)
- **Cyclical**: `month_sin`, `month_cos` - Preserves that December is near January
- **Categorical**: `is_weekend`, `is_quarter_end` - Behavioral indicators

In [9]:
from customer_retention.stages.profiling.temporal_analyzer import TemporalRecommendationType

datetime_cols = [
    name for name, col in findings.columns.items()
    if col.inferred_type == ColumnType.DATETIME
    and name not in TEMPORAL_METADATA_COLS
]

temporal_analyzer = TemporalAnalyzer()

# Store all datetime recommendations grouped by type
feature_engineering_recs = []
modeling_strategy_recs = []
data_quality_recs = []
datetime_summaries = []

for col_name in datetime_cols:
    col_info = findings.columns[col_name]
    print(f"\n{'='*70}")
    print(f"Column: {col_name}")
    print(f"Type: {col_info.inferred_type.value} (Confidence: {col_info.confidence:.0%})")
    print(f"{'='*70}")

    date_series = to_datetime(df[col_name], errors='coerce', format='mixed')
    valid_dates = date_series.dropna()

    print(f"\n📅 Date Range: {valid_dates.min()} to {valid_dates.max()}")
    print(f"   Nulls: {date_series.isna().sum():,} ({date_series.isna().mean()*100:.1f}%)")

    # Basic temporal analysis
    analysis = temporal_analyzer.analyze(date_series)
    print(f"   Auto-detected granularity: {analysis.granularity.value}")
    print(f"   Span: {analysis.span_days:,} days ({analysis.span_days/365:.1f} years)")

    # Growth analysis
    growth = temporal_analyzer.calculate_growth_rate(date_series)
    if growth.get("has_data"):
        print("\n📈 Growth Analysis:")
        print(f"   Trend: {growth['trend_direction'].upper()}")
        print(f"   Overall growth: {growth['overall_growth_pct']:+.1f}%")
        print(f"   Avg monthly growth: {growth['avg_monthly_growth']:+.1f}%")

    # Seasonality analysis
    seasonality = temporal_analyzer.analyze_seasonality(date_series)
    if seasonality.has_seasonality:
        print("\n🔄 Seasonality Detected:")
        print(f"   Peak months: {', '.join(seasonality.peak_periods[:3])}")
        print(f"   Trough months: {', '.join(seasonality.trough_periods[:3])}")
        print(f"   Seasonal strength: {seasonality.seasonal_strength:.2f}")

    # Get recommendations using framework
    other_dates = [c for c in datetime_cols if c != col_name]
    recommendations = temporal_analyzer.recommend_features(date_series, col_name, other_date_columns=other_dates)

    # Group by recommendation type
    col_feature_recs = [r for r in recommendations if r.recommendation_type == TemporalRecommendationType.FEATURE_ENGINEERING]
    col_modeling_recs = [r for r in recommendations if r.recommendation_type == TemporalRecommendationType.MODELING_STRATEGY]
    col_quality_recs = [r for r in recommendations if r.recommendation_type == TemporalRecommendationType.DATA_QUALITY]

    feature_engineering_recs.extend(col_feature_recs)
    modeling_strategy_recs.extend(col_modeling_recs)
    data_quality_recs.extend(col_quality_recs)

    # Display recommendations grouped by type
    if col_feature_recs:
        print("\n🛠️ FEATURES TO CREATE:")
        for rec in col_feature_recs:
            priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "✓"
            print(f"   {priority_icon} {rec.feature_name} ({rec.category})")
            print(f"      Why: {rec.reason}")
            if rec.code_hint:
                print(f"      Code: {rec.code_hint}")

    if col_modeling_recs:
        print("\n⚙️ MODELING CONSIDERATIONS:")
        for rec in col_modeling_recs:
            priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "✓"
            print(f"   {priority_icon} {rec.feature_name}")
            print(f"      Why: {rec.reason}")

    if col_quality_recs:
        print("\n⚠️ DATA QUALITY ISSUES:")
        for rec in col_quality_recs:
            priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "✓"
            print(f"   {priority_icon} {rec.feature_name}")
            print(f"      Why: {rec.reason}")
            if rec.code_hint:
                print(f"      Code: {rec.code_hint}")

    # Standard extractions always available
    print("\n   Standard extractions available: year, month, day, day_of_week, quarter")

    # Store summary
    datetime_summaries.append({
        "Column": col_name,
        "Span (days)": analysis.span_days,
        "Seasonality": "Yes" if seasonality.has_seasonality else "No",
        "Trend": growth.get('trend_direction', 'N/A').capitalize() if growth.get("has_data") else "N/A",
        "Features to Create": len(col_feature_recs),
        "Modeling Notes": len(col_modeling_recs),
        "Quality Issues": len(col_quality_recs)
    })

    # === VISUALIZATIONS ===

    if growth.get("has_data"):
        fig = charts.growth_summary_indicators(growth, title=f"Growth Summary: {col_name}")
        display_figure(fig)

    chart_type = "line" if analysis.granularity in [TemporalGranularity.DAY, TemporalGranularity.WEEK] else "bar"
    fig = charts.temporal_distribution(analysis, title=f"Records Over Time: {col_name}", chart_type=chart_type)
    display_figure(fig)

    fig = charts.temporal_trend(analysis, title=f"Trend Analysis: {col_name}")
    display_figure(fig)

    yoy_data = temporal_analyzer.year_over_year_comparison(date_series)
    if len(yoy_data) > 1:
        fig = charts.year_over_year_lines(yoy_data, title=f"Year-over-Year: {col_name}")
        display_figure(fig)
        fig = charts.year_month_heatmap(yoy_data, title=f"Records Heatmap: {col_name}")
        display_figure(fig)

    if growth.get("has_data"):
        fig = charts.cumulative_growth_chart(growth["cumulative"], title=f"Cumulative Records: {col_name}")
        display_figure(fig)

    fig = charts.temporal_heatmap(date_series, title=f"Day of Week Distribution: {col_name}")
    display_figure(fig)

# === DATETIME SUMMARY ===
if datetime_summaries:
    print("\n" + "=" * 70)
    print("DATETIME COLUMNS SUMMARY")
    print("=" * 70)
    display_table(native_pd.DataFrame(datetime_summaries))

    # Summary by recommendation type
    print("\n📋 ALL RECOMMENDATIONS BY TYPE:")

    if feature_engineering_recs:
        print(f"\n🛠️ FEATURES TO CREATE ({len(feature_engineering_recs)}):")
        for i, rec in enumerate(feature_engineering_recs, 1):
            priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "✓"
            print(f"   {i}. {priority_icon} {rec.feature_name}")

    if modeling_strategy_recs:
        print(f"\n⚙️ MODELING CONSIDERATIONS ({len(modeling_strategy_recs)}):")
        for i, rec in enumerate(modeling_strategy_recs, 1):
            priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "✓"
            print(f"   {i}. {priority_icon} {rec.feature_name}: {rec.reason}")

    if data_quality_recs:
        print(f"\n⚠️ DATA QUALITY TO ADDRESS ({len(data_quality_recs)}):")
        for i, rec in enumerate(data_quality_recs, 1):
            priority_icon = "🔴" if rec.priority == "high" else "🟡" if rec.priority == "medium" else "✓"
            print(f"   {i}. {priority_icon} {rec.feature_name}: {rec.reason}")

    # Add recommendations to registry
    added_derived = 0
    added_modeling = 0

    # Add feature engineering recommendations to Silver layer (derived columns)
    if registry.silver:
        for rec in feature_engineering_recs:
            registry.add_silver_derived(
                column=rec.feature_name,
                expression=rec.code_hint or "",
                feature_type=rec.category,
                rationale=rec.reason,
                source_notebook="04_column_deep_dive"
            )
            added_derived += 1

    # Add modeling strategy recommendations to Bronze layer
    seen_strategies = set()
    for rec in modeling_strategy_recs:
        if rec.feature_name not in seen_strategies:
            registry.add_bronze_modeling_strategy(
                strategy=rec.feature_name,
                column=datetime_cols[0] if datetime_cols else "",
                parameters={"category": rec.category},
                rationale=rec.reason,
                source_notebook="04_column_deep_dive"
            )
            seen_strategies.add(rec.feature_name)
            added_modeling += 1

    print(f"\n✅ Added {added_derived} derived column recommendations to Silver layer")
    print(f"✅ Added {added_modeling} modeling strategy recommendations to Bronze layer")


Column: contract_start
Type: datetime (Confidence: 90%)

📅 Date Range: 2022-02-07 05:36:00+00:00 to 2026-02-03 14:18:00+00:00
   Nulls: 0 (0.0%)
   Auto-detected granularity: quarter
   Span: 1,457 days (4.0 years)

📈 Growth Analysis:
   Trend: GROWING
   Overall growth: -66.7%
   Avg monthly growth: +9.4%

🛠️ FEATURES TO CREATE:
   🟡 days_since_contract_start (recency)
      Why: Recency captures how recent an event occurred - useful for predicting behavior
      Code: (reference_date - df['contract_start']).dt.days
   🟡 days_between_contract_start_and_contract_end (duration)
      Why: Duration between events captures behavioral patterns (e.g., time to convert)
      Code: (df['contract_end'] - df['contract_start']).dt.days
   🟡 days_between_contract_start_and_churn_end (duration)
      Why: Duration between events captures behavioral patterns (e.g., time to convert)
      Code: (df['churn_end'] - df['contract_start']).dt.days
   🟡 tenure_from_contract_start (tenure)
      Why: Long


Column: contract_end
Type: datetime (Confidence: 90%)

📅 Date Range: 2023-02-07 05:36:00+00:00 to 2027-02-03 14:18:00+00:00
   Nulls: 0 (0.0%)
   Auto-detected granularity: quarter
   Span: 1,457 days (4.0 years)

📈 Growth Analysis:
   Trend: GROWING
   Overall growth: -66.7%
   Avg monthly growth: +9.4%

🛠️ FEATURES TO CREATE:
   🟡 days_since_contract_end (recency)
      Why: Recency captures how recent an event occurred - useful for predicting behavior
      Code: (reference_date - df['contract_end']).dt.days
   🟡 days_between_contract_end_and_contract_start (duration)
      Why: Duration between events captures behavioral patterns (e.g., time to convert)
      Code: (df['contract_start'] - df['contract_end']).dt.days
   🟡 days_between_contract_end_and_churn_end (duration)
      Why: Duration between events captures behavioral patterns (e.g., time to convert)
      Code: (df['churn_end'] - df['contract_end']).dt.days
   🟡 tenure_from_contract_end (tenure)
      Why: Long history (4.


Column: churn_end
Type: datetime (Confidence: 90%)

📅 Date Range: 2022-09-13 17:49:00+00:00 to 2026-02-04 10:28:00+00:00
   Nulls: 900 (90.0%)
   Auto-detected granularity: quarter
   Span: 1,239 days (3.4 years)

📈 Growth Analysis:
   Trend: GROWING
   Overall growth: +100.0%
   Avg monthly growth: +23.6%

🔄 Seasonality Detected:
   Peak months: Sep, Jan, Jun
   Trough months: Apr, May, Aug
   Seasonal strength: 0.16

🛠️ FEATURES TO CREATE:
   🟡 days_since_churn_end (recency)
      Why: Recency captures how recent an event occurred - useful for predicting behavior
      Code: (reference_date - df['churn_end']).dt.days
   🟡 days_between_churn_end_and_contract_start (duration)
      Why: Duration between events captures behavioral patterns (e.g., time to convert)
      Code: (df['contract_start'] - df['churn_end']).dt.days
   🟡 days_between_churn_end_and_contract_end (duration)
      Why: Duration between events captures behavioral patterns (e.g., time to convert)
      Code: (df['cont


DATETIME COLUMNS SUMMARY


Column,Span (days),Seasonality,Trend,Features to Create,Modeling Notes,Quality Issues
contract_start,1457,No,Growing,4,1,0
contract_end,1457,No,Growing,4,1,0
churn_end,1239,Yes,Growing,6,1,0



📋 ALL RECOMMENDATIONS BY TYPE:

🛠️ FEATURES TO CREATE (14):
   1. 🟡 days_since_contract_start
   2. 🟡 days_between_contract_start_and_contract_end
   3. 🟡 days_between_contract_start_and_churn_end
   4. 🟡 tenure_from_contract_start
   5. 🟡 days_since_contract_end
   6. 🟡 days_between_contract_end_and_contract_start
   7. 🟡 days_between_contract_end_and_churn_end
   8. 🟡 tenure_from_contract_end
   9. 🟡 days_since_churn_end
   10. 🟡 days_between_churn_end_and_contract_start
   11. 🟡 days_between_churn_end_and_contract_end
   12. 🟡 churn_end_month_sin_cos
   13. 🟡 tenure_from_churn_end
   14. ✓ churn_end_is_weekend

⚙️ MODELING CONSIDERATIONS (3):
   1. 🔴 time_based_train_test_split: Significant growing trend (-67%) detected - random splits would leak future patterns into training
   2. 🔴 time_based_train_test_split: Significant growing trend (-67%) detected - random splits would leak future patterns into training
   3. 🔴 time_based_train_test_split: Significant growing trend (+100%) de

## 4.8 Type Override (Optional)

If any column types were incorrectly inferred, you can override them here.

**Common overrides:**
- Binary columns detected as numeric → `ColumnType.BINARY`
- IDs detected as numeric → `ColumnType.IDENTIFIER`
- Ordinal categories detected as nominal → `ColumnType.CATEGORICAL_ORDINAL`

In [10]:
# === TYPE OVERRIDES ===
# Uncomment and modify to override any incorrectly inferred types
TYPE_OVERRIDES = {
    # "column_name": ColumnType.NEW_TYPE,
    # Examples:
    # "is_active": ColumnType.BINARY,
    # "user_id": ColumnType.IDENTIFIER,
    # "satisfaction_level": ColumnType.CATEGORICAL_ORDINAL,
}

if TYPE_OVERRIDES:
    print("Applying type overrides:")
    for col_name, new_type in TYPE_OVERRIDES.items():
        if col_name in findings.columns:
            old_type = findings.columns[col_name].inferred_type.value
            findings.columns[col_name].inferred_type = new_type
            findings.columns[col_name].confidence = 1.0
            findings.columns[col_name].evidence.append("Manually overridden")
            print(f"  {col_name}: {old_type} → {new_type.value}")
else:
    print("No type overrides configured.")
    print("To override a type, add entries to TYPE_OVERRIDES dictionary above.")

No type overrides configured.
To override a type, add entries to TYPE_OVERRIDES dictionary above.


## 4.9 Data Segmentation Analysis

**Purpose:** Determine if the dataset contains natural subgroups that might benefit from separate models.

**📖 Why This Matters:**
- Some datasets have distinct customer segments with very different behaviors
- A single model might struggle to capture patterns that vary significantly across segments
- Segmented models can improve accuracy but add maintenance complexity

**Recommendations:**
- **single_model** - Data is homogeneous; one model for all records
- **consider_segmentation** - Some variation exists; evaluate if complexity is worth it
- **strong_segmentation** - Distinct segments with different target rates; separate models likely beneficial

**Important:** This is exploratory guidance only. The final decision depends on business context, model complexity tolerance, and available resources.

In [11]:
from customer_retention.core.compat import is_databricks
from customer_retention.stages.profiling import SegmentAnalyzer, SparkSegmentAnalyzer

MAX_SEGMENT_SAMPLE_SIZE = 50_000

if is_databricks():
    segment_analyzer = SparkSegmentAnalyzer(max_sample_size=MAX_SEGMENT_SAMPLE_SIZE)
else:
    segment_analyzer = SegmentAnalyzer()

# Find target column if detected
target_col = None
for col_name, col_info in findings.columns.items():
    if col_info.inferred_type == ColumnType.TARGET:
        target_col = col_name
        break

# Run segmentation analysis using numeric features
print("="*70)
print("DATA SEGMENTATION ANALYSIS")
print("="*70)

segmentation = segment_analyzer.analyze(
    df,
    target_col=target_col,
    feature_cols=numeric_cols if numeric_cols else None,
    max_segments=5
)

print("\n🎯 Analysis Results:")
print(f"   Method: {segmentation.method.value}")
print(f"   Detected Segments: {segmentation.n_segments}")
print(f"   Cluster Quality Score: {segmentation.quality_score:.2f}")
if segmentation.target_variance_ratio is not None:
    print(f"   Target Variance Ratio: {segmentation.target_variance_ratio:.2f}")

print("\n📊 Segment Profiles:")
for profile in segmentation.profiles:
    target_info = f" | Target Rate: {profile.target_rate*100:.1f}%" if profile.target_rate is not None else ""
    print(f"   Segment {profile.segment_id}: {profile.size:,} records ({profile.size_pct:.1f}%){target_info}")

# Display recommendation card
fig = charts.segment_recommendation_card(segmentation)
display_figure(fig)

# Display segment overview
fig = charts.segment_overview(segmentation, title="Segment Overview")
display_figure(fig)

# Display feature comparison if we have features
if segmentation.n_segments > 1 and any(p.defining_features for p in segmentation.profiles):
    fig = charts.segment_feature_comparison(segmentation, title="Feature Comparison Across Segments")
    display_figure(fig)

print("\n📝 Rationale:")
for reason in segmentation.rationale:
    print(f"   • {reason}")

DATA SEGMENTATION ANALYSIS

🎯 Analysis Results:
   Method: kmeans
   Detected Segments: 1
   Cluster Quality Score: 0.00
   Target Variance Ratio: 0.00

📊 Segment Profiles:
   Segment 0: 1 records (100.0%) | Target Rate: 0.0%



📝 Rationale:
   • Insufficient data for meaningful segmentation


## 4.10 Save Updated Findings

In [12]:
# Save updated findings back to the same file
findings.save(FINDINGS_PATH)
print(f"Updated findings saved to: {FINDINGS_PATH}")

# Save recommendations registry
recommendations_path = FINDINGS_PATH.replace("_findings.yaml", "_recommendations.yaml")
registry.save(recommendations_path)
print(f"Recommendations saved to: {recommendations_path}")

# Summary of recommendations
all_recs = registry.all_recommendations
print("\n📋 Recommendations Summary:")
print(f"   Bronze layer: {len(registry.get_by_layer('bronze'))} recommendations")
print(f"   Silver layer: {len(registry.get_by_layer('silver'))} recommendations")
print(f"   Gold layer: {len(registry.get_by_layer('gold'))} recommendations")
print(f"   Total: {len(all_recs)} recommendations")


Updated findings saved to: /Users/Vital/python/CustomerRetention/experiments/runs/3set-e48aeb77/datasets/customer_profiles/findings/customer_profiles_findings.yaml
Recommendations saved to: /Users/Vital/python/CustomerRetention/experiments/runs/3set-e48aeb77/datasets/customer_profiles/findings/customer_profiles_recommendations.yaml

📋 Recommendations Summary:
   Bronze layer: 1 recommendations
   Silver layer: 14 recommendations
   Gold layer: 3 recommendations
   Total: 18 recommendations


---

## Summary: What We Learned

In this notebook, we performed a deep dive analysis that included:

1. **Value Range Validation** - Validated rates, binary fields, and non-negative constraints
2. **Numeric Distribution Analysis** - Calculated skewness, kurtosis, and percentiles with transformation recommendations
3. **Categorical Distribution Analysis** - Calculated imbalance ratio, entropy, and concentration with encoding recommendations
4. **Datetime Analysis** - Analyzed seasonality, trends, and patterns with feature engineering recommendations
5. **Data Segmentation** - Evaluated if natural subgroups exist that might benefit from separate models

## Key Metrics Reference

**Numeric Columns:**
| Metric | Threshold | Action |
|--------|-----------|--------|
| Skewness | \|skew\| > 1 | Log transform |
| Kurtosis | > 10 | Cap outliers first |
| Zero % | > 40% | Zero-inflation handling |

**Categorical Columns:**
| Metric | Threshold | Action |
|--------|-----------|--------|
| Imbalance Ratio | > 10x | Group rare categories |
| Entropy | < 50% | Stratified sampling |
| Rare Categories | > 0 | Group into "Other" |

**Datetime Columns:**
| Finding | Action |
|---------|--------|
| Seasonality | Add cyclical month encoding |
| Strong trend | Time-based train/test split |
| Multiple dates | Calculate duration features |
| Placeholder dates | Filter or flag |

## Transformation & Encoding Summary

Review the summary tables above for:
- **Numeric**: Which columns need log transforms, capping, or zero-inflation handling
- **Categorical**: Which encoding to use and whether to group rare categories
- **Datetime**: Which temporal features to engineer based on detected patterns

---

## Next Steps

Continue to **02_source_integrity.ipynb** to:
- Analyze duplicate records and value conflicts
- Deep dive into missing value patterns
- Analyze outliers with IQR method
- Check data consistency
- Get cleaning recommendations

Or jump to **05_feature_opportunities.ipynb** if you want to see derived feature recommendations.

> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.